#### Imports and API key check

In [2]:
import json
import os
import re
from datetime import date, datetime, timedelta
from pathlib import Path

import requests
from dotenv import load_dotenv

load_dotenv()  # reads the API key from your .env file
DATA_DIR = Path("data")


print("Google key found:", bool(os.getenv("GOOGLE_API_KEY")))
print("data folder found:", DATA_DIR.exists())
print("Files:", [f.name for f in DATA_DIR.glob("*.json")] if DATA_DIR.exists() else "not found")

Google key found: True
data folder found: True
Files: ['flights.json', 'hotels.json', 'places.json']


#### Load the data

In [3]:
def load_json(filename: str) -> list:
    """Load a JSON file from the data folder."""
    try:
        with open(DATA_DIR / filename, encoding="utf-8") as file:
            return json.load(file)
    except (FileNotFoundError, json.JSONDecodeError) as error:
        raise RuntimeError(f"Could not load {filename}: {error}")


flights = load_json("flights.json")
hotels = load_json("hotels.json")
places = load_json("places.json")

print("Flights:", len(flights), "| Hotels:", len(hotels), "| Places:", len(places))
print(flights[0])
print(hotels[0])
print(places[0])

Flights: 30 | Hotels: 40 | Places: 40
{'flight_id': 'FL0001', 'airline': 'IndiGo', 'from': 'Hyderabad', 'to': 'Delhi', 'departure_time': '2025-01-04T11:32:00', 'arrival_time': '2025-01-04T15:32:00', 'price': 2907}
{'hotel_id': 'HOT0001', 'name': 'Grand Palace Hotel', 'city': 'Delhi', 'stars': 4, 'price_per_night': 3897, 'amenities': ['wifi', 'pool']}
{'place_id': 'PLC0001', 'name': 'Famous Fort', 'city': 'Delhi', 'type': 'lake', 'rating': 4.6}


#### Find common cities and routes

In [4]:
# Which cities appear in all three files?
flight_cities = {f["from"] for f in flights} | {f["to"] for f in flights}
hotel_cities = {h["city"] for h in hotels}
place_cities = {p["city"] for p in places}

print("Flight cities:", sorted(flight_cities))
print("Hotel cities :", sorted(hotel_cities))
print("Place cities :", sorted(place_cities))
print("COMMON       :", sorted(flight_cities & hotel_cities & place_cities))

print("\nAvailable routes:")
for route in sorted({(f["from"], f["to"]) for f in flights}):
    print(route)

print("\nPlace types:", sorted({p["type"] for p in places}))

Flight cities: ['Bangalore', 'Chennai', 'Delhi', 'Goa', 'Hyderabad', 'Jaipur', 'Kolkata', 'Mumbai']
Hotel cities : ['Bangalore', 'Chennai', 'Delhi', 'Goa', 'Hyderabad', 'Jaipur', 'Kolkata', 'Mumbai']
Place cities : ['Bangalore', 'Chennai', 'Delhi', 'Goa', 'Hyderabad', 'Jaipur', 'Kolkata', 'Mumbai']
COMMON       : ['Bangalore', 'Chennai', 'Delhi', 'Goa', 'Hyderabad', 'Jaipur', 'Kolkata', 'Mumbai']

Available routes:
('Bangalore', 'Delhi')
('Bangalore', 'Goa')
('Bangalore', 'Kolkata')
('Bangalore', 'Mumbai')
('Chennai', 'Bangalore')
('Chennai', 'Hyderabad')
('Chennai', 'Mumbai')
('Delhi', 'Kolkata')
('Goa', 'Bangalore')
('Goa', 'Hyderabad')
('Goa', 'Jaipur')
('Goa', 'Kolkata')
('Goa', 'Mumbai')
('Hyderabad', 'Delhi')
('Hyderabad', 'Goa')
('Hyderabad', 'Kolkata')
('Hyderabad', 'Mumbai')
('Jaipur', 'Bangalore')
('Jaipur', 'Chennai')
('Jaipur', 'Delhi')
('Jaipur', 'Kolkata')
('Jaipur', 'Mumbai')
('Kolkata', 'Jaipur')
('Mumbai', 'Goa')
('Mumbai', 'Hyderabad')

Place types: ['beach', 'fort', 

#### Flight search function

In [5]:
def search_flights(source: str, destination: str, sort_by: str = "price") -> list:
    """Find flights from source to destination.

    sort_by: 'price' (cheapest first) or 'duration' (fastest first).
    Returns up to 5 flights; empty list if none found.
    """
    matches = []
    for flight in flights:
        if (flight["from"].strip().lower() == source.strip().lower()
                and flight["to"].strip().lower() == destination.strip().lower()):
            departure = datetime.fromisoformat(flight["departure_time"])
            arrival = datetime.fromisoformat(flight["arrival_time"])
            item = dict(flight)
            item["duration_minutes"] = int((arrival - departure).total_seconds() // 60)
            item["departs_at"] = departure.strftime("%H:%M")
            item["arrives_at"] = arrival.strftime("%H:%M")
            matches.append(item)
    key = "duration_minutes" if sort_by == "duration" else "price"
    return sorted(matches, key=lambda f: f[key])[:5]


# Quick test
print("--- Mumbai -> Goa (cheapest first) ---")
for f in search_flights("Mumbai", "Goa", "price"):
    print(f)

print("\n--- Mumbai -> Goa (fastest first) ---")
for f in search_flights("Mumbai", "Goa", "duration"):
    print(f)

print("\n--- Goa -> Mumbai (return) ---")
for f in search_flights("Goa", "Mumbai", "price"):
    print(f)

print("\n--- No such route (should be an empty list) ---")
print(search_flights("Delhi", "Mumbai"))

--- Mumbai -> Goa (cheapest first) ---
{'flight_id': 'FL0024', 'airline': 'SpiceJet', 'from': 'Mumbai', 'to': 'Goa', 'departure_time': '2025-07-15T14:38:00', 'arrival_time': '2025-07-15T18:38:00', 'price': 3304, 'duration_minutes': 240, 'departs_at': '14:38', 'arrives_at': '18:38'}

--- Mumbai -> Goa (fastest first) ---
{'flight_id': 'FL0024', 'airline': 'SpiceJet', 'from': 'Mumbai', 'to': 'Goa', 'departure_time': '2025-07-15T14:38:00', 'arrival_time': '2025-07-15T18:38:00', 'price': 3304, 'duration_minutes': 240, 'departs_at': '14:38', 'arrives_at': '18:38'}

--- Goa -> Mumbai (return) ---
{'flight_id': 'FL0025', 'airline': 'Air India', 'from': 'Goa', 'to': 'Mumbai', 'departure_time': '2025-06-28T16:40:00', 'arrival_time': '2025-06-28T20:40:00', 'price': 7533, 'duration_minutes': 240, 'departs_at': '16:40', 'arrives_at': '20:40'}

--- No such route (should be an empty list) ---
[]


#### Hotel search function

In [6]:
def search_hotels(city: str, min_stars: int = 0, max_price: float = 1_000_000) -> list:
    """Find hotels in a city by minimum stars and max price per night.

    Sorted by stars (high first), then price (low first). Returns up to 5.
    """
    matches = [
        h for h in hotels
        if h["city"].strip().lower() == city.strip().lower()
        and h["stars"] >= min_stars
        and h["price_per_night"] <= max_price
    ]
    return sorted(matches, key=lambda h: (-h["stars"], h["price_per_night"]))[:5]


# Quick test
print("--- All hotels in Goa ---")
for h in search_hotels("Goa"):
    print(h)

print("\n--- Goa, 4+ stars ---")
for h in search_hotels("Goa", min_stars=4):
    print(h)

print("\n--- Goa, under 3000 per night ---")
for h in search_hotels("Goa", max_price=3000):
    print(h)

print("\n--- Unknown city (should be an empty list) ---")
print(search_hotels("Atlantis"))

--- All hotels in Goa ---
{'hotel_id': 'HOT0014', 'name': 'Royal Heritage', 'city': 'Goa', 'stars': 5, 'price_per_night': 1232, 'amenities': ['wifi', 'parking', 'breakfast', 'pool']}
{'hotel_id': 'HOT0011', 'name': 'Comfort Suites', 'city': 'Goa', 'stars': 5, 'price_per_night': 2828, 'amenities': ['spa', 'pool', 'wifi', 'gym']}
{'hotel_id': 'HOT0015', 'name': 'Budget Stay Inn', 'city': 'Goa', 'stars': 4, 'price_per_night': 2248, 'amenities': ['wifi', 'spa', 'pool', 'gym']}
{'hotel_id': 'HOT0012', 'name': 'Green Leaf Resort', 'city': 'Goa', 'stars': 3, 'price_per_night': 2357, 'amenities': ['spa', 'gym', 'pool', 'breakfast']}
{'hotel_id': 'HOT0013', 'name': 'Green Leaf Resort', 'city': 'Goa', 'stars': 2, 'price_per_night': 6259, 'amenities': ['parking', 'breakfast', 'spa', 'pool']}

--- Goa, 4+ stars ---
{'hotel_id': 'HOT0014', 'name': 'Royal Heritage', 'city': 'Goa', 'stars': 5, 'price_per_night': 1232, 'amenities': ['wifi', 'parking', 'breakfast', 'pool']}
{'hotel_id': 'HOT0011', 'nam

#### Places search function

In [7]:
def search_places(city: str, place_type: str = "", limit: int = 8) -> list:
    """Find top-rated attractions in a city, optionally filtered by type."""
    matches = [p for p in places if p["city"].strip().lower() == city.strip().lower()]
    if place_type:
        matches = [p for p in matches if place_type.strip().lower() in p["type"].lower()]
    return sorted(matches, key=lambda p: -p["rating"])[:limit]


# Quick test
print("--- Top places in Goa ---")
for p in search_places("Goa"):
    print(p)

print("\n--- Goa, beaches only ---")
for p in search_places("Goa", place_type="beach"):
    print(p)

print("\n--- Goa, top 3 only ---")
for p in search_places("Goa", limit=3):
    print(p)

print("\n--- Unknown city (should be an empty list) ---")
print(search_places("Atlantis"))

--- Top places in Goa ---
{'place_id': 'PLC0012', 'name': 'Famous Park', 'city': 'Goa', 'type': 'museum', 'rating': 4.5}
{'place_id': 'PLC0014', 'name': 'Beautiful Park', 'city': 'Goa', 'type': 'fort', 'rating': 4.3}
{'place_id': 'PLC0011', 'name': 'Popular Lake', 'city': 'Goa', 'type': 'museum', 'rating': 4.2}
{'place_id': 'PLC0013', 'name': 'Historic Park', 'city': 'Goa', 'type': 'fort', 'rating': 4.1}
{'place_id': 'PLC0015', 'name': 'Beautiful Lake', 'city': 'Goa', 'type': 'market', 'rating': 4.0}

--- Goa, beaches only ---

--- Goa, top 3 only ---
{'place_id': 'PLC0012', 'name': 'Famous Park', 'city': 'Goa', 'type': 'museum', 'rating': 4.5}
{'place_id': 'PLC0014', 'name': 'Beautiful Park', 'city': 'Goa', 'type': 'fort', 'rating': 4.3}
{'place_id': 'PLC0011', 'name': 'Popular Lake', 'city': 'Goa', 'type': 'museum', 'rating': 4.2}

--- Unknown city (should be an empty list) ---
[]


#### Weather function

In [8]:
GEOCODING_URL = "https://geocoding-api.open-meteo.com/v1/search"
FORECAST_URL = "https://api.open-meteo.com/v1/forecast"
MAX_FORECAST_DAYS = 15

WEATHER_CODES = {
    0: "Clear sky", 1: "Mainly clear", 2: "Partly cloudy", 3: "Overcast",
    45: "Fog", 48: "Fog", 51: "Light drizzle", 53: "Drizzle", 55: "Heavy drizzle",
    61: "Light rain", 63: "Rain", 65: "Heavy rain", 71: "Light snow", 73: "Snow",
    80: "Rain showers", 81: "Rain showers", 82: "Heavy rain showers",
    95: "Thunderstorm", 96: "Thunderstorm with hail", 99: "Thunderstorm with hail",
}


def get_coordinates(city: str) -> tuple:
    """Return (latitude, longitude) for a city using Open-Meteo geocoding."""
    try:
        response = requests.get(GEOCODING_URL, params={"name": city, "count": 1}, timeout=10)
        response.raise_for_status()
        result = response.json()["results"][0]
        return result["latitude"], result["longitude"]
    except (requests.RequestException, KeyError, IndexError) as error:
        raise ValueError(f"Could not find coordinates for '{city}': {error}")


def get_weather(city: str, start_date: str, end_date: str) -> list:
    """Get daily weather forecast for a city. Dates must be YYYY-MM-DD."""
    try:
        start = date.fromisoformat(start_date)
        end = date.fromisoformat(end_date)
    except ValueError:
        return [{"error": "Dates must be in YYYY-MM-DD format."}]
    if end < start:
        return [{"error": "End date is before start date."}]
    if start > date.today() + timedelta(days=MAX_FORECAST_DAYS):
        return [{"error": f"Forecast not available yet (only next {MAX_FORECAST_DAYS} days)."}]

    try:
        latitude, longitude = get_coordinates(city)
        params = {
            "latitude": latitude, "longitude": longitude,
            "daily": "temperature_2m_max,temperature_2m_min,weathercode,precipitation_probability_max",
            "start_date": start_date, "end_date": end_date, "timezone": "auto",
        }
        response = requests.get(FORECAST_URL, params=params, timeout=10)
        response.raise_for_status()
        daily = response.json()["daily"]
    except (ValueError, requests.RequestException, KeyError) as error:
        return [{"error": f"Weather unavailable: {error}"}]

    return [
        {
            "date": daily["time"][i],
            "max_temp_c": daily["temperature_2m_max"][i],
            "min_temp_c": daily["temperature_2m_min"][i],
            "condition": WEATHER_CODES.get(daily["weathercode"][i], "Unknown"),
            "rain_chance_percent": daily["precipitation_probability_max"][i],
        }
        for i in range(len(daily["time"]))
    ]


# Quick test
trip_start = date.today() + timedelta(days=3)
trip_end = trip_start + timedelta(days=2)

print(f"Today: {date.today()}")
print(f"--- Goa weather, {trip_start} to {trip_end} ---")
for day in get_weather("Goa", str(trip_start), str(trip_end)):
    print(day)

print("\n--- Too far in the future (should show an error message, not crash) ---")
print(get_weather("Goa", "2030-01-01", "2030-01-03"))

print("\n--- End before start (should show an error message) ---")
print(get_weather("Goa", str(trip_end), str(trip_start)))

print("\n--- Unknown city (should show an error message) ---")
print(get_weather("Zzzzxxqq", str(trip_start), str(trip_end)))

Today: 2026-11-09
--- Goa weather, 2026-11-12 to 2026-11-14 ---
{'date': '2026-11-12', 'max_temp_c': 14.8, 'min_temp_c': 13.0, 'condition': 'Rain showers', 'rain_chance_percent': 96}
{'date': '2026-11-13', 'max_temp_c': 16.7, 'min_temp_c': 12.8, 'condition': 'Light rain', 'rain_chance_percent': 83}
{'date': '2026-11-14', 'max_temp_c': 16.7, 'min_temp_c': 11.7, 'condition': 'Partly cloudy', 'rain_chance_percent': 18}

--- Too far in the future (should show an error message, not crash) ---
[{'error': 'Forecast not available yet (only next 15 days).'}]

--- End before start (should show an error message) ---
[{'error': 'End date is before start date.'}]

--- Unknown city (should show an error message) ---
[{'error': "Weather unavailable: Could not find coordinates for 'Zzzzxxqq': 'results'"}]


In [9]:
CITY_COORDINATES = {
    "bangalore": (12.9716, 77.5946),
    "chennai": (13.0827, 80.2707),
    "delhi": (28.6139, 77.2090),
    "goa": (15.2993, 74.1240),
    "hyderabad": (17.3850, 78.4867),
    "jaipur": (26.9124, 75.7873),
    "kolkata": (22.5726, 88.3639),
    "mumbai": (19.0760, 72.8777),
}


def get_coordinates(city: str) -> tuple:
    """Return (latitude, longitude) for a city.

    Uses the built-in table for dataset cities first, then falls back to
    Open-Meteo geocoding restricted to India.
    """
    known = CITY_COORDINATES.get(city.strip().lower())
    if known:
        return known
    try:
        response = requests.get(
            GEOCODING_URL,
            params={"name": city, "count": 1, "countryCode": "IN"},
            timeout=10,
        )
        response.raise_for_status()
        result = response.json()["results"][0]
        return result["latitude"], result["longitude"]
    except (requests.RequestException, KeyError, IndexError) as error:
        raise ValueError(f"Could not find coordinates for '{city}': {error}")


# Re-test
trip_start = date.today() + timedelta(days=3)
trip_end = trip_start + timedelta(days=2)

print(f"--- Goa weather, {trip_start} to {trip_end} ---")
for day in get_weather("Goa", str(trip_start), str(trip_end)):
    print(day)

print("\n--- Mumbai weather (second city check) ---")
for day in get_weather("Mumbai", str(trip_start), str(trip_end)):
    print(day)

print("\n--- Unknown city (should still show an error message) ---")
print(get_weather("Zzzzxxqq", str(trip_start), str(trip_end)))

--- Goa weather, 2026-11-12 to 2026-11-14 ---
{'date': '2026-11-12', 'max_temp_c': 35.7, 'min_temp_c': 19.9, 'condition': 'Overcast', 'rain_chance_percent': 0}
{'date': '2026-11-13', 'max_temp_c': 35.6, 'min_temp_c': 19.3, 'condition': 'Clear sky', 'rain_chance_percent': 0}
{'date': '2026-11-14', 'max_temp_c': 35.7, 'min_temp_c': 20.0, 'condition': 'Overcast', 'rain_chance_percent': 2}

--- Mumbai weather (second city check) ---
{'date': '2026-11-12', 'max_temp_c': 34.6, 'min_temp_c': 23.5, 'condition': 'Clear sky', 'rain_chance_percent': 0}
{'date': '2026-11-13', 'max_temp_c': 34.7, 'min_temp_c': 23.5, 'condition': 'Clear sky', 'rain_chance_percent': 0}
{'date': '2026-11-14', 'max_temp_c': 34.4, 'min_temp_c': 22.6, 'condition': 'Clear sky', 'rain_chance_percent': 0}

--- Unknown city (should still show an error message) ---
[{'error': "Weather unavailable: Could not find coordinates for 'Zzzzxxqq': 'results'"}]


#### Budget function

In [10]:
def estimate_budget(outbound_flight_price: float, hotel_price_per_night: float, nights: int,
                    return_flight_price: float = 0, daily_expense: float = 800,
                    travelers: int = 1) -> dict:
    """Calculate total trip cost.

    Flight = (outbound + return) * travelers
    Hotel  = price per night * nights (per room)
    Food and local travel = daily_expense * (nights + 1) * travelers
    """
    flight_total = (outbound_flight_price + return_flight_price) * travelers
    hotel_total = hotel_price_per_night * nights
    local_total = daily_expense * (nights + 1) * travelers
    return {
        "flight": flight_total,
        "hotel": hotel_total,
        "food_and_local_travel": local_total,
        "total": flight_total + hotel_total + local_total,
    }


# Quick test using real numbers from your data:
# Mumbai -> Goa flight 3304, Goa -> Mumbai flight 7533, Royal Heritage hotel 1232/night, 2 nights
print("--- Real trip: Mumbai -> Goa, 2 nights, 1 traveler ---")
print(estimate_budget(3304, 1232, 2, return_flight_price=7533))

print("\n--- Same trip, 2 travelers ---")
print(estimate_budget(3304, 1232, 2, return_flight_price=7533, travelers=2))

--- Real trip: Mumbai -> Goa, 2 nights, 1 traveler ---
{'flight': 10837, 'hotel': 2464, 'food_and_local_travel': 2400, 'total': 15701}

--- Same trip, 2 travelers ---
{'flight': 21674, 'hotel': 2464, 'food_and_local_travel': 4800, 'total': 28938}


#### Wrap the functions as LangChain tools

In [11]:
from langchain_core.tools import tool


@tool
def flight_search(source: str, destination: str, sort_by: str = "price") -> list:
    """Search one-way flights between two cities. sort_by is 'price' (cheapest) or 'duration' (fastest). Call twice (once each way) for a round trip."""
    return search_flights(source, destination, sort_by)


@tool
def hotel_search(city: str, min_stars: int = 0, max_price: float = 1000000) -> list:
    """Find hotels in a city. Filter by minimum stars and maximum price per night."""
    return search_hotels(city, min_stars, max_price)


@tool
def places_search(city: str, place_type: str = "") -> list:
    """Find top-rated tourist attractions in a city. Optionally filter by place type (beach, fort, lake, market, monument, museum, park, temple)."""
    return search_places(city, place_type)


@tool
def weather_lookup(city: str, start_date: str, end_date: str) -> list:
    """Get the daily weather forecast for a city. Dates must be YYYY-MM-DD."""
    return get_weather(city, start_date, end_date)


@tool
def budget_estimator(outbound_flight_price: float, hotel_price_per_night: float, nights: int,
                     return_flight_price: float = 0, daily_expense: float = 800,
                     travelers: int = 1) -> dict:
    """Estimate total trip cost from flight prices, hotel price per night, nights and daily local expenses."""
    return estimate_budget(outbound_flight_price, hotel_price_per_night, nights,
                           return_flight_price, daily_expense, travelers)


ALL_TOOLS = [flight_search, hotel_search, places_search, weather_lookup, budget_estimator]

print("Tools ready:", [t.name for t in ALL_TOOLS])

# Test: call each tool the same way the agent will
print("\n--- flight_search ---")
print(flight_search.invoke({"source": "Mumbai", "destination": "Goa"}))

print("\n--- hotel_search ---")
print(hotel_search.invoke({"city": "Goa", "min_stars": 4}))

print("\n--- places_search ---")
print(places_search.invoke({"city": "Goa", "place_type": "fort"}))

print("\n--- budget_estimator ---")
print(budget_estimator.invoke({
    "outbound_flight_price": 3304, "hotel_price_per_night": 1232,
    "nights": 2, "return_flight_price": 7533,
}))

print("\n--- Tool description the agent will read ---")
print(flight_search.name, "->", flight_search.description)
print(flight_search.args)

Tools ready: ['flight_search', 'hotel_search', 'places_search', 'weather_lookup', 'budget_estimator']

--- flight_search ---
[{'flight_id': 'FL0024', 'airline': 'SpiceJet', 'from': 'Mumbai', 'to': 'Goa', 'departure_time': '2025-07-15T14:38:00', 'arrival_time': '2025-07-15T18:38:00', 'price': 3304, 'duration_minutes': 240, 'departs_at': '14:38', 'arrives_at': '18:38'}]

--- hotel_search ---
[{'hotel_id': 'HOT0014', 'name': 'Royal Heritage', 'city': 'Goa', 'stars': 5, 'price_per_night': 1232, 'amenities': ['wifi', 'parking', 'breakfast', 'pool']}, {'hotel_id': 'HOT0011', 'name': 'Comfort Suites', 'city': 'Goa', 'stars': 5, 'price_per_night': 2828, 'amenities': ['spa', 'pool', 'wifi', 'gym']}, {'hotel_id': 'HOT0015', 'name': 'Budget Stay Inn', 'city': 'Goa', 'stars': 4, 'price_per_night': 2248, 'amenities': ['wifi', 'spa', 'pool', 'gym']}]

--- places_search ---
[{'place_id': 'PLC0014', 'name': 'Beautiful Park', 'city': 'Goa', 'type': 'fort', 'rating': 4.3}, {'place_id': 'PLC0013', 'name'

#### System prompt, Gemini model, and the agent

In [16]:
SYSTEM_PROMPT = """You are an expert travel planner. Build a complete trip plan using ONLY the provided tools.

Process:
1. flight_search for the outward flight AND the return flight (swap source and destination). Pick the cheapest by default, the fastest if the user asks.
2. hotel_search in the destination within the user's budget. Pick the best option (high stars, fair price).
3. places_search for attractions in the destination.
4. weather_lookup for the travel dates.
5. budget_estimator for the cost breakdown. Nights = number of days minus 1.
6. Make a day-wise itinerary. Group places sensibly, do not overload a day, adapt to weather.

Rules:
- Never invent flights, hotels or places. Use only what the tools return.
- If a tool returns no data, say so clearly instead of guessing.
- If the weather tool returns an error, mention that the forecast is unavailable.
- The flight data has fixed dates from the past. Ignore those dates and use the user's travel dates.
- If the total exceeds the user's budget, say so and suggest cheaper options.

Final answer format:
## Trip Summary
## Flight Selected
## Hotel Recommendation
## Day-wise Itinerary (with weather for each day)
## Budget Breakdown (with total)
## Why I chose these

After the readable plan, add a ```json block with keys:
trip_summary, flight, hotel, itinerary, weather, budget, reasoning.
"""

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

llm = init_chat_model("gemini-3.8-flash", model_provider="google_genai", temperature=0)

agent = create_agent(model=llm, tools=ALL_TOOLS, system_prompt=SYSTEM_PROMPT)


def plan_trip(query: str, show_steps: bool = True) -> str:
    """Run the agent on a travel request and return the final answer."""
    try:
        result = agent.invoke({"messages": [{"role": "user", "content": query}]})
    except Exception as error:
        return f"Agent error: {error}"

    if show_steps:  # shows which tools the agent decided to call
        for message in result["messages"]:
            for call in getattr(message, "tool_calls", []) or []:
                print("Tool call:", call["name"], call["args"])
    return result["messages"][-1].content


# Quick test 1: is the Gemini key working?
print("--- Model check ---")
print(llm.invoke("Reply with just the word: ready").content)

print("\nAgent created successfully.")

--- Model check ---
[{'type': 'text', 'text': 'ready', 'extras': {'signature': 'EuADCt0DAWkUfRMzFzXpbQfBdeyLp4+Hji17iuMEADoAq9re7pXOnwwQ1YYJekGRT3VKLigQU+PvCAh6FfFPeraw6mdxiDmVN+pt46+BZG5glX9c2893ot+Alpp5vv8Vlx2kp8ofb4ADJWtvJ7y0ez0ci/RgLv8Ssl4ywSU64/xeELNuXQQc2d4ms5zZXB25iuXxMuBT3Dw4wXg+anrL3NV1K+XydWf06rWFa0K1aQn5GEBjUUFrqAqak9/1OHDKqeig5yOq56EbejKice8CZ/5lw78jOtm27HnjxdFmD+DKQsU5EY+owpefbUgz76UntYO7TzndmVOMMW08TbC6jPVFWGADPkhSuj7m7ZDk9I/RHohcswNh1oChAQBJBsDnh8AXjuTeCw9gboOzHQCwFt5CRN0bdxoazs8sF9PeHecntYhK8pJb90pL+KUlzfun1REjKjSJpzpRoYK2phUjoczoX6adyLu/4wvg/5mvnRKcE691owXM+Y4wy8N7GEoQOyiox6PhGj2gYOcHJgSUaCCQVpqAgQEwWiCAXRHJtqpYHPCPfaKloi+3ip4/NxzfHoNfdxEQCKd1wZJwJf5yaEzgZldx8p9+SwOEzQXsHv+2DHa7Bh2I+FQR/TOupR4uc887'}}]

Agent created successfully.


In [18]:
for name in ["init_chat_model", "create_agent", "ALL_TOOLS", "SYSTEM_PROMPT", "flights"]:
    print(name, "->", "OK" if name in globals() else "MISSING")


init_chat_model -> MISSING
create_agent -> MISSING
ALL_TOOLS -> OK
SYSTEM_PROMPT -> MISSING
flights -> OK


In [19]:
import time
from IPython.display import Markdown, display
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

SYSTEM_PROMPT = """You are an expert travel planner. Build a complete trip plan using ONLY the provided tools.

Process:
1. flight_search for the outward flight AND the return flight (swap source and destination). Pick the cheapest by default, the fastest if the user asks.
2. hotel_search in the destination within the user's budget. Pick the best option (high stars, fair price).
3. places_search for attractions in the destination.
4. weather_lookup for the travel dates.
5. budget_estimator for the cost breakdown. Nights = number of days minus 1.
6. Make a day-wise itinerary. Group places sensibly, do not overload a day, adapt to weather.

Rules:
- Never invent flights, hotels or places. Use only what the tools return.
- If a tool returns no data, say so clearly instead of guessing.
- If the weather tool returns an error, mention that the forecast is unavailable.
- The flight data has fixed dates from the past. Ignore those dates and use the user's travel dates.
- If the total exceeds the user's budget, say so and suggest cheaper options.

Final answer format:
## Trip Summary
## Flight Selected
## Hotel Recommendation
## Day-wise Itinerary (with weather for each day)
## Budget Breakdown (with total)
## Why I chose these

After the readable plan, add a ```json block with keys:
trip_summary, flight, hotel, itinerary, weather, budget, reasoning.
"""

# Check that everything Cell B needs now exists
for name in ["init_chat_model", "create_agent", "ALL_TOOLS", "SYSTEM_PROMPT",
             "flights", "date", "timedelta", "time", "Markdown", "display"]:
    print(name, "->", "OK" if name in globals() else "MISSING")

init_chat_model -> OK
create_agent -> OK
ALL_TOOLS -> OK
SYSTEM_PROMPT -> OK
flights -> OK
date -> OK
timedelta -> OK
time -> OK
Markdown -> OK
display -> OK


In [22]:
%pip install -q -U langchain-xai

from dotenv import load_dotenv
load_dotenv(override=True)  # re-read .env so the new key is picked up

import os, time, requests
from langchain_xai import ChatXAI

print("XAI key found:", bool(os.getenv("XAI_API_KEY")))

# 1. Ask the xAI API which models this key can use
response = requests.get(
    "https://api.x.ai/v1/models",
    headers={"Authorization": f"Bearer {os.getenv('XAI_API_KEY')}"},
    timeout=20,
)
print("Status:", response.status_code)
if response.ok:
    model_ids = sorted(m["id"] for m in response.json()["data"])
    print("Models:", model_ids)
else:
    print(response.text[:300])


[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
XAI key found: True
Status: 400
{"code":"invalid-argument","error":"Incorrect API key provided. You can obtain an API key from https://console.x.ai."}


In [23]:


from dotenv import load_dotenv
load_dotenv(override=True)  # re-read .env so the new key is picked up

import os, requests

key = os.getenv("GROQ_API_KEY") or ""
print("Groq key found:", bool(key), "| starts with:", repr(key[:4]), "| length:", len(key))

# Ask the Groq API which models this key can use
response = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers={"Authorization": f"Bearer {key}"},
    timeout=20,
)
print("Status:", response.status_code)
if response.ok:
    print("Models:", sorted(m["id"] for m in response.json()["data"]))
else:
    print(response.text[:300])

Groq key found: True | starts with: 'gsk_' | length: 56
Status: 200
Models: ['allam-2-7b', 'canopylabs/orpheus-arabic-saudi', 'canopylabs/orpheus-v1-english', 'meta-llama/llama-prompt-guard-2-22m', 'meta-llama/llama-prompt-guard-2-86m', 'openai/gpt-oss-120b', 'openai/gpt-oss-20b', 'openai/gpt-oss-safeguard-20b', 'qwen/qwen3.8-27b', 'whisper-large-v3', 'whisper-large-v3-turbo']


#### Groq model, test call, and your first real trip

In [25]:
import json
from pathlib import Path

SYSTEM_PROMPT_V2 = SYSTEM_PROMPT + """

Additional rules:
- Do NOT write weekday names (Mon, Tue, ...). Use dates only, like 12 Nov 2026.
- Use the 'arrives_at' time from the outbound flight result as the arrival time on Day 1. Plan only activities AFTER that time. If arrival is after 17:00, Day 1 is only check-in and dinner.
- Use the 'departs_at' time of the return flight on the last day. Plan nothing within 3 hours before that departure except check-out and airport transfer.
- Hotels have 'stars', not ratings. Places have 'rating'.
- Do not invent restaurant or attraction names. Generic meal suggestions are fine.
- Put the JSON block inside a ```json code fence.
"""

agent = create_agent(model=llm, tools=ALL_TOOLS, system_prompt=SYSTEM_PROMPT_V2)

start = time.time()
result = agent.invoke(
    {"messages": [{"role": "user", "content": query}]},
    config={"recursion_limit": 25},
)
print(f"Total time: {time.time() - start:.1f}s")
text = content_to_text(result["messages"][-1].content)


def extract_json(answer: str):
    """Return the trip JSON from the agent's answer as a dict, or None."""
    fenced = re.search(r"```json\s*(\{.*\})\s*```", answer, re.DOTALL)
    if fenced:
        candidate = fenced.group(1)
    else:  # fall back: find the object that starts at "trip_summary"
        marker = answer.find('"trip_summary"')
        brace = answer.rfind("{", 0, marker) if marker != -1 else -1
        if brace == -1:
            return None
        candidate = answer[brace:]
    try:
        obj, _ = json.JSONDecoder().raw_decode(candidate)
        return obj
    except json.JSONDecodeError:
        return None


plan_json = extract_json(text)

display(Markdown(text))
print("\n" + "=" * 60)

if plan_json:
    Path("trip_plan.json").write_text(
        json.dumps(plan_json, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    print("JSON keys:", list(plan_json.keys()))
    print("Budget total in JSON:", plan_json["budget"]["total"])
    print("Saved to trip_plan.json")
else:
    print("Could not extract JSON from the answer.")

Total time: 61.0s


## Trip Summary
- **Travel dates:** 12 Nov 2026 – 14 Nov 2026 (2 nights)  
- **Destination:** Goa, India  
- **Travelers:** 1  
- **Budget:** 25 000 INR  

## Flight Selected
| Leg | Airline | Flight ID | Departs | Arrives | Price (INR) |
|-----|---------|-----------|---------|---------|-------------|
| Outbound (Mumbai → Goa) | SpiceJet | FL0024 | 14:38 (12 Nov) | 18:38 (12 Nov) | 3 304 |
| Return (Goa → Mumbai) | Air India | FL0025 | 16:40 (14 Nov) | 20:40 (14 Nov) | 7 533 |

*Cheapest options were chosen as requested.*

## Hotel Recommendation
- **Royal Heritage** – 5 ★, 1,232 INR per night  
- Amenities: Wi‑Fi, parking, breakfast, pool  
- Total for 2 nights: 2 464 INR  

## Day‑wise Itinerary (with weather)

| Date | Weather (max/min °C) | Activities |
|------|----------------------|------------|
| **12 Nov 2026** (arrival day) | Overcast, 35.7/19.9, no rain | 18:38 – Arrive in Goa (flight FL0024)  <br> 19:15 – Transfer to Royal Heritage, check‑in  <br> 20:00 – Light dinner (hotel or nearby restaurant)  <br> Evening free – relax at hotel |
| **13 Nov 2026** | Clear sky, 35.6/19.3, no rain | 08:30 – Breakfast at hotel  <br> 09:30 – Visit **Beautiful Park** (fort, rating 4.3)  <br> 12:30 – Lunch (local eatery)  <br> 14:00 – Explore **Famous Park** (museum, rating 4.5)  <br> 16:30 – Optional stop at **Historic Park** (fort, rating 4.1)  <br> 19:00 – Dinner (hotel or recommended restaurant) |
| **14 Nov 2026** (departure day) | Overcast, 35.7/20.0, 2 % rain chance | 08:00 – Breakfast and check‑out  <br> 09:00 – Visit **Popular Lake** (museum‑type attraction, rating 4.2)  <br> 11:30 – Return to hotel, collect luggage  <br> 12:00 – Light lunch  <br> 12:30 – Transfer to Goa airport  <br> 13:40 – Arrive at airport (3 h before departure)  <br> 16:40 – Depart Goa (flight FL0025) |

## Budget Breakdown
| Item | Cost (INR) |
|------|------------|
| Flights (both ways) | 10 837 |
| Hotel (2 nights) | 2 464 |
| Food & local travel (₹800 × 3 days) | 2 400 |
| **Total** | **15 701** |

*The total stays well within the 25 000 INR budget, leaving ~9 300 INR for souvenirs or extra meals.*

## Why I Chose These Options
- **Flights:** The SpiceJet outbound and Air India return were the cheapest flights returned by the search, matching the priority for cost.
- **Hotel:** Royal Heritage offers the highest star rating (5 ★) at the lowest nightly price among 5‑star options, maximizing comfort while staying budget‑friendly.
- **Attractions:** All forts and museums from the places search are included, ordered to avoid travel fatigue and to fit the weather forecast (clear sky on the full‑day of 13 Nov).
- **Weather Consideration:** Activities are scheduled outdoors on the clear‑sky day; indoor museum visits are placed on overcast days.
- **Budget:** The estimator shows the trip costs ~15 701 INR, comfortably under the 25 000 INR limit, allowing flexibility for extra expenses.

---  

```json
{
  "trip_summary": "2‑night trip to Goa (12‑14 Nov 2026) with cheapest flights, 5‑star budget hotel, forts and museums, total cost 15,701 INR.",
  "flight": {
    "outbound": {
      "airline": "SpiceJet",
      "flight_id": "FL0024",
      "departure_time": "14:38",
      "arrival_time": "18:38",
      "price": 3304
    },
    "return": {
      "airline": "Air India",
      "flight_id": "FL0025",
      "departure_time": "16:40",
      "arrival_time": "20:40",
      "price": 7533
    }
  },
  "hotel": {
    "hotel_id": "HOT0014",
    "name": "Royal Heritage",
    "stars": 5,
    "price_per_night": 1232,
    "nights": 2,
    "total_price": 2464
  },
  "itinerary": [
    {
      "date": "2026-11-12",
      "weather": "Overcast, max 35.7°C, min 19.9°C",
      "activities": [
        "18:38 Arrive Goa (flight FL0024)",
        "19:15 Transfer to hotel, check‑in",
        "20:00 Dinner",
        "Evening free"
      ]
    },
    {
      "date": "2026-11-13",
      "weather": "Clear sky, max 35.6°C, min 19.3°C",
      "activities": [
        "08:30 Breakfast at hotel",
        "09:30 Visit Beautiful Park (fort)",
        "12:30 Lunch",
        "14:00 Visit Famous Park (museum)",
        "16:30 Optional Historic Park (fort)",
        "19:00 Dinner"
      ]
    },
    {
      "date": "2026-11-14",
      "weather": "Overcast, max 35.7°C, min 20.0°C",
      "activities": [
        "08:00 Breakfast, check‑out",
        "09:00 Visit Popular Lake (museum‑type)",
        "11:30 Return to hotel, collect luggage",
        "12:00 Light lunch",
        "12:30 Transfer to airport",
        "13:40 Arrive airport",
        "16:40 Depart Goa (flight FL0025)"
      ]
    }
  ],
  "weather": [
    {
      "date": "2026-11-12",
      "max_temp_c": 35.7,
      "min_temp_c": 19.9,
      "condition": "Overcast",
      "rain_chance_percent": 0
    },
    {
      "date": "2026-11-13",
      "max_temp_c": 35.6,
      "min_temp_c": 19.3,
      "condition": "Clear sky",
      "rain_chance_percent": 0
    },
    {
      "date": "2026-11-14",
      "max_temp_c": 35.7,
      "min_temp_c": 20.0,
      "condition": "Overcast",
      "rain_chance_percent": 2
    }
  ],
  "budget": {
    "flight": 10837,
    "hotel": 2464,
    "food_and_local_travel": 2400,
    "total": 15701,
    "budget_limit": 25000,
    "remaining": 9299
  },
  "reasoning": "Cheapest flights were selected per user priority. Royal Heritage gives 5‑star comfort at the lowest price among high‑rated hotels, keeping overall cost low. All forts and museums from the places list are included, arranged to match weather conditions and arrival/departure times. The estimated total (15,701 INR) stays well under the 25,000 INR budget, providing a comfortable margin for extra spending."
}
```


JSON keys: ['trip_summary', 'flight', 'hotel', 'itinerary', 'weather', 'budget', 'reasoning']
Budget total in JSON: 15701
Saved to trip_plan.json


In [26]:
def run_scenario(name: str, scenario_query: str, preview_chars: int = 1500) -> None:
    """Run one test scenario and print tool calls plus a short preview of the answer."""
    print("=" * 70)
    print("SCENARIO:", name)
    print("QUERY   :", scenario_query)
    print("=" * 70)
    try:
        result = agent.invoke(
            {"messages": [{"role": "user", "content": scenario_query}]},
            config={"recursion_limit": 25},
        )
    except Exception as error:
        print("CRASHED :", str(error)[:300])
        return

    for message in result["messages"]:
        for call in getattr(message, "tool_calls", []) or []:
            print("  tool ->", call["name"], call["args"])

    text = content_to_text(result["messages"][-1].content)
    text = text.split("```json")[0]  # hide the JSON block for a shorter preview
    print("\nANSWER PREVIEW:\n", text[:preview_chars])
    print()


today = date.today()
soon = today + timedelta(days=3)

scenarios = {
    "1. Very low budget": (
        f"Plan a trip from Mumbai to Goa, from {soon} to {soon + timedelta(days=2)}. "
        "Budget 2000 INR. Priority: cheapest flight."
    ),
    "2. City with no data": (
        f"Plan a trip from Mumbai to Paris, from {soon} to {soon + timedelta(days=2)}. "
        "Budget 50000 INR."
    ),
    "3. Route with no flight": (
        f"Plan a trip from Delhi to Mumbai, from {soon} to {soon + timedelta(days=2)}. "
        "Budget 30000 INR."
    ),
    "4. End date before start date": (
        f"Plan a trip from Mumbai to Goa, from {soon + timedelta(days=3)} to {soon}. "
        "Budget 25000 INR."
    ),
    "5. Dates too far for weather": (
        "Plan a trip from Mumbai to Goa, from 2027-03-10 to 2027-03-12. "
        "Budget 25000 INR."
    ),
}

for scenario_name, scenario_query in scenarios.items():
    run_scenario(scenario_name, scenario_query)
    time.sleep(20)  # avoid the free-tier rate limit

SCENARIO: 1. Very low budget
QUERY   : Plan a trip from Mumbai to Goa, from 2026-11-12 to 2026-11-14. Budget 2000 INR. Priority: cheapest flight.
  tool -> flight_search {'destination': 'Goa', 'sort_by': 'price', 'source': 'Mumbai'}
  tool -> flight_search {'destination': 'Mumbai', 'sort_by': 'price', 'source': 'Goa'}

ANSWER PREVIEW:
 **Result:**  
The only flights returned by the system are:

| Direction | Flight ID | Airline   | Departs (local) | Arrives (local) | Price (INR) |
|-----------|-----------|-----------|-----------------|-----------------|-------------|
| Mumbai → Goa | FL0024 | SpiceJet | 14:38 | 18:38 | **3,304** |
| Goa → Mumbai | FL0025 | Air India | 16:40 | 20:40 | **7,533** |

**Why this matters:**  
Your total budget is **2,000 INR**. Even the cheapest outbound flight alone (3,304 INR) already exceeds that amount, and the round‑trip cost (≈ 10,837 INR) is far above the budget.

### What can be done
- **Increase the budget** to at least cover the flight cost (≈ 3,30

In [27]:
def run_scenario_short(name: str, scenario_query: str, keywords: list) -> str:
    """Run one scenario, print tool calls and a short preview, and return the full answer."""
    print("=" * 60)
    print("SCENARIO:", name)
    try:
        result = agent.invoke(
            {"messages": [{"role": "user", "content": scenario_query}]},
            config={"recursion_limit": 25},
        )
    except Exception as error:
        print("CRASHED :", str(error)[:200])
        return f"## {name}\nCRASHED: {error}\n"

    tools_used = [call["name"] for m in result["messages"]
                  for call in (getattr(m, "tool_calls", None) or [])]
    text = content_to_text(result["messages"][-1].content)
    body = text.split("```json")[0]

    print("Tools called :", tools_used)
    found = [k for k in keywords if k.lower() in body.lower()]
    print("Keywords seen:", found if found else "NONE of " + str(keywords))
    print("Preview      :", body[:500].replace("\n", " "), "...\n")
    return f"## {name}\n\nTools: {tools_used}\n\n{body}\n\n---\n"


soon = date.today() + timedelta(days=3)
checks = {
    "2. City with no data": (
        f"Plan a trip from Mumbai to Paris, from {soon} to {soon + timedelta(days=2)}. Budget 50000 INR.",
        ["no data", "not available", "no flight", "no hotel", "unable", "cannot", "couldn't"],
    ),
    "3. Route with no flight": (
        f"Plan a trip from Delhi to Mumbai, from {soon} to {soon + timedelta(days=2)}. Budget 30000 INR.",
        ["no flight", "no direct", "not available", "unable", "cannot", "couldn't"],
    ),
    "4. End date before start date": (
        f"Plan a trip from Mumbai to Goa, from {soon + timedelta(days=3)} to {soon}. Budget 25000 INR.",
        ["before", "invalid", "earlier", "check", "correct"],
    ),
    "5. Dates too far for weather": (
        "Plan a trip from Mumbai to Goa, from 2027-03-10 to 2027-03-12. Budget 25000 INR.",
        ["forecast", "unavailable", "not available"],
    ),
}

all_output = "# Edge-case test results\n\n"
for scenario_name, (scenario_query, words) in checks.items():
    all_output += run_scenario_short(scenario_name, scenario_query, words)
    time.sleep(20)  # avoid the free-tier rate limit

Path("test_results.md").write_text(all_output, encoding="utf-8")
print("Full answers saved to test_results.md")

SCENARIO: 2. City with no data
CRASHED : Error code: 400 - {'error': {'message': "'messages.3' : for 'role:tool' the following must be satisfied[('messages.3.content' : one of the following must be satisfied[('messages.3.content' : value mus
SCENARIO: 3. Route with no flight
CRASHED : Error code: 400 - {'error': {'message': "'messages.3' : for 'role:tool' the following must be satisfied[('messages.3.content' : one of the following must be satisfied[('messages.3.content' : value mus
SCENARIO: 4. End date before start date
Tools called : ['flight_search', 'flight_search', 'hotel_search', 'places_search', 'weather_lookup', 'budget_estimator']
Keywords seen: ['before', 'check']
Preview      : ## Trip Summary - **Travelers:** 1   - **Travel dates:** 12 Nov 2026 – 15 Nov 2026 (3 nights)   - **Destination:** Goa, India   - **Budget:** ₹25,000    ## Flight Selected | Leg | Airline | Flight ID | Departs (local) | Arrives (local) | Price (INR) | |-----|---------|-----------|-----------------|---

In [28]:
from langchain_core.tools import tool


@tool
def flight_search(source: str, destination: str, sort_by: str = "price") -> list:
    """Search one-way flights between two cities. sort_by is 'price' (cheapest) or 'duration' (fastest). Call twice (once each way) for a round trip."""
    results = search_flights(source, destination, sort_by)
    return results or f"No flights found from {source} to {destination} in the dataset."


@tool
def hotel_search(city: str, min_stars: int = 0, max_price: float = 1000000) -> list:
    """Find hotels in a city. Filter by minimum stars and maximum price per night."""
    results = search_hotels(city, min_stars, max_price)
    return results or f"No hotels found in {city} for these filters."


@tool
def places_search(city: str, place_type: str = "") -> list:
    """Find top-rated tourist attractions in a city. Optionally filter by place type (beach, fort, lake, market, monument, museum, park, temple)."""
    results = search_places(city, place_type)
    return results or f"No places found in {city} for type '{place_type}'."


@tool
def weather_lookup(city: str, start_date: str, end_date: str) -> list:
    """Get the daily weather forecast for a city. Dates must be YYYY-MM-DD."""
    return get_weather(city, start_date, end_date)


@tool
def budget_estimator(outbound_flight_price: float, hotel_price_per_night: float, nights: int,
                     return_flight_price: float = 0, daily_expense: float = 800,
                     travelers: int = 1) -> dict:
    """Estimate total trip cost from flight prices, hotel price per night, nights and daily local expenses."""
    return estimate_budget(outbound_flight_price, hotel_price_per_night, nights,
                           return_flight_price, daily_expense, travelers)


ALL_TOOLS = [flight_search, hotel_search, places_search, weather_lookup, budget_estimator]

SYSTEM_PROMPT_V3 = SYSTEM_PROMPT_V2 + f"""

Input validation (do this BEFORE calling any tool). Today's date is {date.today()}.
- If the end date is before the start date, or the start date is in the past, do NOT call any tool and do NOT swap or fix the dates yourself. Reply only with a short message explaining the problem and asking the user for corrected dates.
- If a tool says no flights, hotels or places were found, tell the user clearly what is missing and do not make up a plan. Suggest a nearby city or route that is likely to work.
- If the user's budget is below the cost of the cheapest flights, say so and stop.
"""

llm = ChatGroq(model=MODEL_NAME, temperature=0, timeout=90, max_retries=1,
               max_tokens=4000, reasoning_effort="low")
agent = create_agent(model=llm, tools=ALL_TOOLS, system_prompt=SYSTEM_PROMPT_V3)


def run_check(name: str, scenario_query: str) -> str:
    """Run one scenario and print tool calls and a short preview."""
    print("=" * 60)
    print("SCENARIO:", name)
    try:
        result = agent.invoke({"messages": [{"role": "user", "content": scenario_query}]},
                              config={"recursion_limit": 25})
    except Exception as error:
        print("  CRASHED:", str(error)[:300])
        return f"## {name}\nCRASHED: {error}\n\n"
    used = [c["name"] for m in result["messages"] for c in (getattr(m, "tool_calls", None) or [])]
    body = content_to_text(result["messages"][-1].content).split("```json")[0]
    print("  Tools called:", used)
    print("  Preview:", body[:450].replace("\n", " "), "...\n")
    return f"## {name}\n\nTools: {used}\n\n{body}\n\n---\n"


soon = date.today() + timedelta(days=3)
fixed = {
    "2. Paris (no data)": f"Plan a trip from Mumbai to Paris, from {soon} to {soon + timedelta(days=2)}. Budget 50000 INR.",
    "3. Delhi to Mumbai (no flight)": f"Plan a trip from Delhi to Mumbai, from {soon} to {soon + timedelta(days=2)}. Budget 30000 INR.",
    "4. End before start": f"Plan a trip from Mumbai to Goa, from {soon + timedelta(days=3)} to {soon}. Budget 25000 INR.",
}

report = "# Edge-case retest\n\n"
for name, q in fixed.items():
    report += run_check(name, q)
    time.sleep(20)

Path("test_results_retest.md").write_text(report, encoding="utf-8")
print("Saved to test_results_retest.md")

SCENARIO: 2. Paris (no data)
  Tools called: ['flight_search']
  Preview: I’m sorry, but the flight search returned no available flights from **Mumbai** to **Paris** in the data set. Without a flight option, I can’t create a complete travel plan for those dates and budget.  Would you like me to look for flights to a nearby city (e.g., London, Brussels, or Amsterdam) or consider a different travel date? Let me know how you’d like to proceed. ...

SCENARIO: 3. Delhi to Mumbai (no flight)
  Tools called: ['flight_search']
  Preview: I’m sorry, but the flight search returned no available flights from Delhi to Mumbai in the dataset, so I can’t create a complete itinerary with the given parameters. You might consider choosing a different destination or checking alternative travel dates. ...

SCENARIO: 4. End before start
  Tools called: []
  Preview: Your travel dates are inconsistent—the end date (2026‑11‑12) is before the start date (2026‑11‑15). Could you please provide a corrected date r